In [ ]:
# Initialize the Adam optimizer
optimizer = optim.Adam(model.parameters(), lr = learning_rate, weight_decay = 1e-6)
# Loop through epochs for training
for epoch in range(1, num_epochs + 1):
    model.train() # Set model for training
    # Rest of your training loop and clear the gradiant
    # Forward pass
    optimizer.zero_grad()
    loss = 0
    p = np.random.permutation(num_Students) # Train the model on whole dataset for different batch size
    # Iterate over the batch of the student
    for student_idx in p[:batch_size]:
        # Get the tensor for X and Y for the chosen student
        x_data = X_train[student_idx].to(device)
        y_data = Y_train[student_idx].to(device)
        outputs = model(x_data)  # Pass x_data through our model
        
        # check the performance for the mean loos for each epoch:
        for t in range(y_data.shape[0]):
            loss = loss + criterion(outputs[t, y_data[t, 0]], y_data[t, 1].float())#.double()) # Loss per timestep
    # Backpropagation: model weights updating
    loss.backward()
    optimizer.step()
    # Calculate validation loss 
    # Validate the model
    model.eval() # set the model for the evalution
    val_loss = 0
    with torch.no_grad(): #  Prevents gradient calculations and, consequently, the weight updates within that block
        for idx in range(len(X_vali)):
            x_val_data = X_vali[idx].to(device)
            y_val_data = Y_vali[idx].to(device)
            val_outputs = model(x_val_data)
             # Calculate the validation loss for each timestep
            for t in range(y_val_data.shape[0]):
                val_loss += criterion(val_outputs[t, y_val_data[t, 0]], y_val_data[t, 1].float())
    # Average the validation loss
    val_loss /= len(X_vali)
    # Print loss at specific epochs
    if epoch % 10 == 0:
        print(f"Epoch {epoch}: Loss {loss.item()}")
        print(f"Epoch {epoch}: Validation Loss {val_loss.item()}")
        loss_values[learning_rate].append([epoch, loss.item(), val_loss.item()])

In [ ]:
Accuracy = 0
for i in range(len(X_test)):
    data     = X_test[i]
    Y_data   = Y_test[i]
    with torch.no_grad():
        data = data.to(device)
        output_probs = model(data)
        #print(output_probs)
        test_loss = 0
        for t in range(output_probs.shape[0]):
            if Y_data[t, 1]>0.5 and output_probs[t,Y_data[t,0]]< 0.5:
                test_loss += 1
            elif Y_data[t, 1]<0.5 and output_probs[t,Y_data[t,0]]> 0.5:
                test_loss +=1
        Accuracy += 1 - test_loss/output_probs.shape[0]
print(Accuracy/ len(X_test))